# Part 3: ResNet50 Permutation Hardness Analysis

This notebook reuses the Part 1 ResNet50 baseline results and saved tile permutations to compute permutation hardness metrics without retraining.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

In [ ]:
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').exists():
        ROOT = candidate
        break
else:
    ROOT = current

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [ ]:
def install_project_requirements_for_colab(project_root: Path) -> None:
    """Install non-PyTorch dependencies in Colab without replacing CUDA-matched torch wheels."""
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return

    import subprocess

    requirements_path = project_root / 'requirements.txt'
    filtered_requirements = Path('/tmp/mlds_colab_requirements.txt')
    skip_prefixes = ('torch', 'torchvision')
    filtered_lines = [
        line
        for line in requirements_path.read_text().splitlines()
        if not line.strip().lower().startswith(skip_prefixes)
    ]
    filtered_requirements.write_text('\n'.join(filtered_lines) + '\n')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(filtered_requirements)])


In [ ]:
install_project_requirements_for_colab(ROOT)

ROOT

In [ ]:
import pandas as pd
from IPython.display import Image, display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)
load_part1_resnet50_results = experiment_results.load_part1_resnet50_results
load_part3_results = experiment_results.load_part3_results
part3_output_paths = experiment_results.part3_output_paths
run_part3_hardness_analysis = experiment_results.run_part3_hardness_analysis

## Configuration
Define Part 3 settings directly in the notebook.

In [ ]:
results_dir = ROOT / 'outputs/results'
figures_dir = ROOT / 'outputs/figures'
part1_results_csv = results_dir / 'part1_raw_results.csv'
permutation_csv = results_dir / 'part1_permutations.csv'

grid_sizes = [1, 2, 3, 4]
num_permutations = 2
seed = 42
alpha_center = 1.0
weight_adj = 0.5
weight_center = 0.3
weight_dist = 0.2

output_paths = part3_output_paths(str(results_dir), str(figures_dir))
pd.DataFrame([
    {
        'grid_sizes': grid_sizes,
        'num_permutations': num_permutations,
        'seed': seed,
        'alpha_center': alpha_center,
        'weight_adj': weight_adj,
        'weight_center': weight_center,
        'weight_dist': weight_dist,
    }
])

## Data Loading
Load existing Part 1 results filtered to the trained ResNet50 baseline.

In [ ]:
part1_resnet50_results = load_part1_resnet50_results(str(part1_results_csv))
print(f'Part 1 ResNet50 result rows: {len(part1_resnet50_results)}')
display(part1_resnet50_results.head())

## Compute Hardness Metrics
Compute permutation-only metrics, join them with ResNet50 accuracy, and save outputs.

In [ ]:
analysis_results = run_part3_hardness_analysis(
    results_dir=str(results_dir),
    figures_dir=str(figures_dir),
    part1_results_csv=str(part1_results_csv),
    permutation_csv=str(permutation_csv),
    grid_sizes=grid_sizes,
    num_permutations=num_permutations,
    seed=seed,
    alpha_center=alpha_center,
    weight_adj=weight_adj,
    weight_center=weight_center,
    weight_dist=weight_dist,
)
display(analysis_results['metrics'].head())
display(analysis_results['joined'].head())

## Correlations
Display Pearson and Spearman correlations between each hardness metric and ResNet50 validation accuracy.

In [ ]:
saved_results = load_part3_results(str(results_dir))
display(saved_results['correlations'])

## Metric Plots
Display saved metric-vs-accuracy plots.

In [ ]:
output_paths = part3_output_paths(str(results_dir), str(figures_dir))
for figure_path in output_paths['plots']:
    display(Image(filename=figure_path))
if not output_paths['plots']:
    print('No Part 3 plots found yet.')